In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import date

fake = Faker()
np.random.seed(42)
random.seed(42)

def generate_cc_risk_model(n=1000):
    data = []
    lgd_map = list("ABCDEFGHIJ") # A = Lowest Severity, J = Highest Severity

    for _ in range(n):
        # 1. Demographic & Profile Data
        p = fake.profile()
        age = date.today().year - p['birthdate'].year
        
        # 2. Exposure At Default (EAD) Calculation
        # EAD = Current Balance + (Unused Limit * CCF)
        limit = np.random.choice([1500, 5000, 10000, 25000])
        balance = np.random.uniform(0, limit * 0.95)
        ccf = 0.25  # 25% of unused limit likely used just before default
        ead = balance + ((limit - balance) * ccf)
        
        # 3. PD Grade Logic (1-15)
        # Higher utilization and lower age typically correlate with higher PD
        utilization = balance / limit
        pd_score = int(np.clip((utilization * 12) + (max(0, 45 - age) / 10), 1, 15))
        
        # 4. LGD Class Logic (A-J)
        # Logic: Smaller balances are easier to recover (lower LGD)
        # Larger exposures often involve higher losses/lower recovery rates
        lgd_base = int(np.clip((ead / 5000) * 2.5 + np.random.randint(0, 4), 0, 9))
        lgd_class = lgd_map[lgd_base]
        
        data.append({
            "Cardholder": p['name'],
            "Age": age,
            "Limit": f"${limit:,}",
            "EAD": round(ead, 2),
            "Utilization": f"{utilization:.1%}",
            "PD_Grade": pd_score,
            "LGD_Class": lgd_class,
            "Risk_Rating": f"{pd_score}{lgd_class}"
        })
        
    return pd.DataFrame(data)

# Generate and view the top 10 exposures
df = generate_cc_risk_model(1000)
print("### TOP CREDIT CARD EXPOSURES & RISK RATINGS")
print(df.sort_values("EAD", ascending=False).head(10).to_string(index=False))

### TOP CREDIT CARD EXPOSURES & RISK RATINGS
       Cardholder  Age   Limit      EAD Utilization  PD_Grade LGD_Class Risk_Rating
     Justin Davis   12 $25,000 23963.77       94.5%        14         J         14J
Denise Richardson   41 $25,000 23923.77       94.3%        11         J         11J
   Christine Paul   78 $25,000 23800.41       93.6%        11         J         11J
         Anne Lee    0 $25,000 23728.27       93.2%        15         J         15J
     Susan Phelps   84 $25,000 23668.86       92.9%        11         J         11J
   Kelly Gonzalez   96 $25,000 23602.31       92.5%        11         J         11J
       Holly Ball   35 $25,000 23595.02       92.5%        12         J         12J
   Barbara Flores   52 $25,000 23483.79       91.9%        11         J         11J
    Caitlin Jones   73 $25,000 23446.53       91.7%        11         J         11J
     Vincent Cruz  109 $25,000 23325.55       91.1%        10         J         10J
